# NJOY Input Deck Builder

Build NJOY input decks programmatically using the fluent `InputDeck` API.

Tape numbers are **auto-chained** by default: `moder` sets the ENDF tape, and each subsequent module automatically reads from the previous output and allocates a new output tape. You can always override with explicit values.

In [1]:
from kika.njoy import InputDeck

## Auto-Chained Tape Numbers

Tape numbers flow automatically through the processing chain — no need to track them manually.

In [2]:
# Minimal auto-chain: moder → reconr → broadr → acer
deck_auto = (
    InputDeck()
    .moder(nin=20)
    .reconr(isotope="U235", err=0.001)
    .broadr(isotope="U235", temperatures=[293.6])
    .acer(isotope="U235", tempd=293.6, suff=0.03)
)
print(deck_auto.render())

moder
20 -21
-- reconstruct, linearise and unionize data
reconr
-21 -22
'reconstructed data for U235 @ 0.0 K'/
9228 /
0.001 /
0 /
-- calculate doppler broadening
broadr
-21 -22 -23
9228 1 /
0.001 /
293.6 /
0 /
-- generate ACE file
acer
-21 -23 0 40 41
1 1 1 0.03 /
'U235 @ 293.6 K ACE data' /
9228 293.6 /
/
/
stop


In [3]:
# Verify auto-chain matches equivalent manual deck
deck_manual = (
    InputDeck()
    .moder(nin=20, nout=-21)
    .reconr(nendf=-21, npend=-22, isotope="U235", err=0.001)
    .broadr(nendf=-21, nin=-22, nout=-23, isotope="U235", temperatures=[293.6])
    .acer(nendf=-21, npend=-23, isotope="U235", tempd=293.6, suff=0.03)
)
assert deck_auto.render() == deck_manual.render(), "Auto-chain output differs from manual!"
print("Auto-chain matches manual deck OK")

Auto-chain matches manual deck OK


In [4]:
# Multi-module auto-chain: Fe56 with unresr + purr
deck_fe56_auto = (
    InputDeck()
    .moder(nin=20)
    .reconr(isotope="Fe56")
    .broadr(isotope="Fe56", temperatures=[293.6, 600.0, 900.0])
    .unresr(isotope="Fe56", temperatures=[293.6, 600.0, 900.0], sigz=[1e10, 1e4, 1e3, 1e2, 1e1])
    .purr(isotope="Fe56", temperatures=[293.6, 600.0, 900.0], sigz=[1e10, 1e4, 1e3, 1e2, 1e1])
)
print(deck_fe56_auto.render())

moder
20 -21
-- reconstruct, linearise and unionize data
reconr
-21 -22
'reconstructed data for Fe56 @ 0.0 K'/
2631 /
0.001 /
0 /
-- calculate doppler broadening
broadr
-21 -22 -23
2631 3 /
0.001 /
293.6 600.0 900.0 /
0 /
-- calculate self-shielding
unresr
-21 -23 -24
2631 3 5 /
293.6 600.0 900.0 /
10000000000.0 10000.0 1000.0 100.0 10.0 /
0 /
-- calculate probability tables
purr
-21 -24 -25
2631 3 5 20 64 /
293.6 600.0 900.0 /
10000000000.0 10000.0 1000.0 100.0 10.0 /
0 /
stop


In [5]:
# Error handling: calling reconr without moder or explicit nendf
try:
    InputDeck().reconr(isotope="U235")
    print("ERROR: should have raised ValueError")
except ValueError as e:
    print(f"OK: {e}")

OK: No ENDF tape available. Call the preceding module first or pass the tape number explicitly.


## Manual Tape Numbers

You can still pass tape numbers explicitly — explicit values always override auto-chaining.

### Minimal Example -- Single Temperature

A basic processing chain for U-235 at room temperature: convert the ENDF tape, reconstruct cross sections, Doppler-broaden, and generate an ACE file.

In [6]:
deck = (
    InputDeck()
    .moder(nin=20, nout=-25)
    .reconr(nendf=-25, npend=-21, isotope="U235", err=0.001)
    .broadr(nendf=-25, nin=-21, nout=-22, isotope="U235", temperatures=[293.6])
    .acer(nendf=-25, npend=-22, isotope="U235", tempd=293.6, suff=0.03)
)
deck

InputDeck(4 modules: MODER -> RECONR -> BROADR -> ACER)

In [7]:
print(deck.render())

moder
20 -25
-- reconstruct, linearise and unionize data
reconr
-25 -21
'reconstructed data for U235 @ 0.0 K'/
9228 /
0.001 /
0 /
-- calculate doppler broadening
broadr
-25 -21 -22
9228 1 /
0.001 /
293.6 /
0 /
-- generate ACE file
acer
-25 -22 0 40 41
1 1 1 0.03 /
'U235 @ 293.6 K ACE data' /
9228 293.6 /
/
/
stop


### Multiple Temperatures and Dilutions

Process Fe-56 at three temperatures with five background cross sections, using `broadr`, `unresr`, and `purr` for full self-shielding treatment.

In [8]:
deck_fe56 = (
    InputDeck()
    .moder(nin=20, nout=-25)
    .reconr(nendf=-25, npend=-21, isotope="Fe56")
    .broadr(
        nendf=-25, nin=-21, nout=-22, isotope="Fe56",
        temperatures=[293.6, 600.0, 900.0],
    )
    .unresr(
        nendf=-25, nin=-22, nout=-23, isotope="Fe56",
        temperatures=[293.6, 600.0, 900.0],
        sigz=[1e10, 1e4, 1e3, 1e2, 1e1],
    )
    .purr(
        nendf=-25, nin=-23, nout=-24, isotope="Fe56",
        temperatures=[293.6, 600.0, 900.0],
        sigz=[1e10, 1e4, 1e3, 1e2, 1e1],
    )
)
print(deck_fe56.render())

moder
20 -25
-- reconstruct, linearise and unionize data
reconr
-25 -21
'reconstructed data for Fe56 @ 0.0 K'/
2631 /
0.001 /
0 /
-- calculate doppler broadening
broadr
-25 -21 -22
2631 3 /
0.001 /
293.6 600.0 900.0 /
0 /
-- calculate self-shielding
unresr
-25 -22 -23
2631 3 5 /
293.6 600.0 900.0 /
10000000000.0 10000.0 1000.0 100.0 10.0 /
0 /
-- calculate probability tables
purr
-25 -23 -24
2631 3 5 20 64 /
293.6 600.0 900.0 /
10000000000.0 10000.0 1000.0 100.0 10.0 /
0 /
stop


### Heat Production and Gas Production

Add KERMA-factor calculation (`heatr`) with multiple MT reactions and gas-production cross sections (`gaspr`).

In [9]:
deck_heat = (
    InputDeck()
    .moder(nin=20, nout=-25)
    .reconr(nendf=-25, npend=-21, isotope="U235")
    .broadr(nendf=-25, nin=-21, nout=-22, isotope="U235", temperatures=[293.6])
    .heatr(nendf=-25, nin=-22, nout=-23, isotope="U235", mtk=[302, 303, 304, 318])
    .gaspr(nendf=-25, nin=-23, nout=-24)
)
print(deck_heat.render())

moder
20 -25
-- reconstruct, linearise and unionize data
reconr
-25 -21
'reconstructed data for U235 @ 0.0 K'/
9228 /
0.001 /
0 /
-- calculate doppler broadening
broadr
-25 -21 -22
9228 1 /
0.001 /
293.6 /
0 /
-- calculate heating values
heatr
-25 -22 -23 0
9228 4 /
302 303 304 318 /
-- calculate gas production
gaspr
-25 -23 -24
stop


### Full Processing Chain

A production-style 8-module chain for U-238 covering reconstruction, broadening, probability tables, heating, gas production, ACE generation, and plot output.

In [10]:
deck_full = (
    InputDeck()
    .moder(nin=20, nout=-25)
    .reconr(nendf=-25, npend=-21, isotope="U238")
    .broadr(nendf=-25, nin=-21, nout=-22, isotope="U238", temperatures=[293.6])
    .purr(
        nendf=-25, nin=-22, nout=-23, isotope="U238",
        temperatures=[293.6], sigz=[1e10],
    )
    .heatr(
        nendf=-25, nin=-23, nout=-24, isotope="U238",
        mtk=[302, 303, 304, 318], nplot=30,
    )
    .gaspr(nendf=-25, nin=-24, nout=-26)
    .acer(
        nendf=-25, npend=-26, isotope="U238",
        tempd=293.6, suff=0.38, nace=40, ndir=41,
    )
    .viewr(infile=30, nps=35)
)
deck_full

InputDeck(8 modules: MODER -> RECONR -> BROADR -> PURR -> HEATR -> GASPR -> ACER -> VIEWR)

In [11]:
print(deck_full.render())

moder
20 -25
-- reconstruct, linearise and unionize data
reconr
-25 -21
'reconstructed data for U238 @ 0.0 K'/
9237 /
0.001 /
0 /
-- calculate doppler broadening
broadr
-25 -21 -22
9237 1 /
0.001 /
293.6 /
0 /
-- calculate probability tables
purr
-25 -22 -23
9237 1 1 20 64 /
293.6 /
10000000000.0 /
0 /
-- calculate heating values
heatr
-25 -23 -24 30
9237 4 /
302 303 304 318 /
-- calculate gas production
gaspr
-25 -24 -26
-- generate ACE file
acer
-25 -26 0 40 41
1 1 1 0.38 /
'U238 @ 293.6 K ACE data' /
9237 293.6 /
/
/
-- produce plots
viewr
30 35
stop


In [12]:
deck_full
assert str(deck_full) == deck_full.render()

# Save to file (uncomment to write):
# deck_full.save("u238_full.inp")

## Thermal Scattering (iopt=2)

Generate thermal scattering ACE data for H in light water. Uses ACER `iopt=2` with the required thermal parameters: ZAID name, moderator ZA values, and the inelastic MT number.

In [13]:
# Thermal scattering — H in light water
deck_thermal = (
    InputDeck()
    .moder(nin=20, nout=-25)
    .reconr(nendf=-25, npend=-21, isotope="H1")
    .broadr(nendf=-25, nin=-21, nout=-22, isotope="H1", temperatures=[293.6])
    .acer(
        nendf=-25, npend=-22, isotope="H1",
        tempd=293.6, suff=0.00, iopt=2,
        tname="lwtr", iza=[1001], mti=222,
    )
)
print(deck_thermal.render())

moder
20 -25
-- reconstruct, linearise and unionize data
reconr
-25 -21
'reconstructed data for H1 @ 0.0 K'/
125 /
0.001 /
0 /
-- calculate doppler broadening
broadr
-25 -21 -22
125 1 /
0.001 /
293.6 /
0 /
-- generate ACE file
acer
-25 -22 0 40 41
2 1 1 0.00 /
'H1 @ 293.6 K ACE data' /
125 293.6 'lwtr' 1 /
1001 /
222 /
stop


In [14]:
# Verify iopt=3 raises ValueError
try:
    InputDeck().acer(iopt=3)
    print("ERROR: should have raised ValueError")
except ValueError as e:
    print(f"OK: {e}")

OK: No ENDF tape available. Call the preceding module first or pass the tape number explicitly.


## Custom VIEWR Plots

Define inline plot commands using the `ViewrPlot` dataclass API. When `plot_def` is provided, VIEWR emits the full card sequence for custom 2D and 3D graphs.

In [15]:
from kika.njoy.viewr_defs import (
    ViewrPlot, PageSetup, Plot, Curve, DataPoint2D,
    CurveStyle, AxisRange,
    Orientation, PlotType, GridStyle, LegendStyle, CurveColor,
)

# Single 2D plot — log-log cross section curve
plot_def = ViewrPlot(
    page=PageSetup(orientation=Orientation.LANDSCAPE),
    plots=[
        Plot(
            title1="U-235 Total Cross Section",
            plot_type=PlotType.LOG_LOG,
            grid=GridStyle.TIC_MARKS_INSIDE,
            x_range=AxisRange(min=1e-5, max=2e7),
            x_label="Energy (eV)",
            y_range=AxisRange(min=0.1, max=1e4),
            y_label="Cross Section (barns)",
            curves=[
                Curve(
                    style=CurveStyle(iccol=CurveColor.BLUE, ithick=2),
                    data_2d=[
                        DataPoint2D(1e-5, 1000.0),
                        DataPoint2D(1e-3, 500.0),
                        DataPoint2D(1e0, 50.0),
                        DataPoint2D(1e3, 10.0),
                        DataPoint2D(2e7, 5.0),
                    ],
                ),
            ],
        ),
    ],
)

deck_viewr = InputDeck().viewr(nps=91, plot_def=plot_def)
print(deck_viewr.render())

-- produce plots
viewr
5 91
1 2 1.0 0 /
1 0 1.0 1.0 0.0 0.0 0.0 0.0 /
'U-235 Total Cross Section'/
''/
4 0 3 0 /
1e-05 20000000.0 0.0 /
'Energy (eV)'/
0.1 10000.0 0.0 /
'Cross Section (barns)'/
0 /
0 0 0 3 2 0 /
0 /
1e-05 1000.0
0.001 500.0
1.0 50.0
1000.0 10.0
20000000.0 5.0
/
99 /
stop


In [16]:
# Multiple curves with legend box
plot_multi = ViewrPlot(
    page=PageSetup(orientation=Orientation.PORTRAIT),
    plots=[
        Plot(
            title1="Comparison of Cross Sections",
            title2="U-235 vs U-238",
            plot_type=PlotType.LOG_LOG,
            legend_style=LegendStyle.LEGEND_BOX,
            x_label="Energy (eV)",
            y_label="Cross Section (barns)",
            curves=[
                Curve(
                    style=CurveStyle(iccol=CurveColor.BLUE, ithick=2),
                    legend="U-235 Total",
                    data_2d=[DataPoint2D(1e-5, 1000.0), DataPoint2D(2e7, 5.0)],
                ),
                Curve(
                    style=CurveStyle(
                        iccol=CurveColor.RED, idash=1, ithick=2,
                    ),
                    legend="U-238 Total",
                    data_2d=[DataPoint2D(1e-5, 12.0), DataPoint2D(2e7, 6.0)],
                ),
            ],
        ),
    ],
)

deck_multi = InputDeck().viewr(nps=91, plot_def=plot_multi)
print(deck_multi.render())

-- produce plots
viewr
5 91
0 2 1.0 0 /
1 0 1.0 1.0 0.0 0.0 0.0 0.0 /
'Comparison of Cross Sections'/
'U-235 vs U-238'/
4 0 3 1 /
0.0 0.0 0.0 /
'Energy (eV)'/
0.0 0.0 0.0 /
'Cross Section (barns)'/
0 /
0 0 0 3 2 0 /
'U-235 Total'/
0 /
1e-05 1000.0
20000000.0 5.0
/
1 0 1.0 1.0 0.0 0.0 0.0 0.0 /
0 /
0 0 1 1 2 0 /
'U-238 Total'/
0 /
1e-05 12.0
20000000.0 6.0
/
99 /
stop


In [17]:
# Backward compatibility — viewr without plot_def still works
deck_compat = (
    InputDeck()
    .moder(nin=20, nout=-25)
    .reconr(nendf=-25, npend=-21, isotope="U238")
    .heatr(nendf=-25, nin=-21, nout=-22, isotope="U238", mtk=[302], nplot=30)
    .viewr(infile=30, nps=35)
)
output = deck_compat.render()
# The viewr section should be exactly 3 lines: comment, module name, unit numbers
viewr_lines = output.split("\n")[-4:-1]  # last 3 lines before "stop"
assert viewr_lines == ["-- produce plots", "viewr", "30 35"], f"Got: {viewr_lines}"
print("Backward compatibility OK")
print(output)

Backward compatibility OK
moder
20 -25
-- reconstruct, linearise and unionize data
reconr
-25 -21
'reconstructed data for U238 @ 0.0 K'/
9237 /
0.001 /
0 /
-- calculate heating values
heatr
-25 -21 -22 30
9237 1 /
302 /
-- produce plots
viewr
30 35
stop


## Summary

- **Isotope names** like `"U235"`, `"Fe56"`, `"U238"` are resolved to MAT numbers automatically
- **Temperature and dilution lists** are passed directly; `ntemp` / `nsigz` counts are handled internally
- **Fluent chaining** lets you build an entire processing pipeline in a single expression
- `render()` returns the deck as a string; `save(path)` writes it to disk
- **Custom VIEWR plots** can be defined with `ViewrPlot` dataclasses and passed via `plot_def=...`
- All 10 NJOY modules are supported: `moder`, `reconr`, `broadr`, `unresr`, `purr`, `heatr`, `gaspr`, `groupr`, `acer`, `viewr`

## GROUPR — Multigroup Cross Sections

Generate multigroup cross sections and scattering matrices from ENDF/PENDF data.

In [18]:
# Basic GROUPR with predefined LANL 30-group (ign=3) and 1/E weight (iwt=3)
deck_groupr = (
    InputDeck()
    .moder(nin=20)
    .reconr(isotope="U235")
    .broadr(isotope="U235", temperatures=[293.6])
    .groupr(
        isotope="U235",
        ign=3,
        iwt=3,
        lord=5,
        temperatures=[293.6],
        sigz=[1e10, 1e4, 1e3, 1e2, 1e1],
        reactions=[(3, 1, "total"), (3, 2, "elastic"), (3, 18, "fission")],
    )
)
print(deck_groupr.render())

moder
20 -21
-- reconstruct, linearise and unionize data
reconr
-21 -22
'reconstructed data for U235 @ 0.0 K'/
9228 /
0.001 /
0 /
-- calculate doppler broadening
broadr
-21 -22 -23
9228 1 /
0.001 /
293.6 /
0 /
-- compute multigroup cross sections
groupr
-21 -23 0 -24
9228 3 0 3 5 1 5 /
'multigroup data for U235'/
293.6 /
10000000000.0 10000.0 1000.0 100.0 10.0 /
3 1 'total' /
3 2 'elastic' /
3 18 'fission' /
0 /
0 /
stop


In [19]:
# Custom neutron group boundaries (ign=1 with egn list)
deck_custom_groups = (
    InputDeck()
    .moder(nin=20)
    .reconr(isotope="U235")
    .broadr(isotope="U235", temperatures=[293.6])
    .groupr(
        isotope="U235",
        ign=1,
        iwt=3,
        lord=3,
        temperatures=[293.6],
        sigz=[1e10],
        egn=[1e-5, 0.625, 5.53e3, 8.21e5, 2e7],
    )
)
print(deck_custom_groups.render())

moder
20 -21
-- reconstruct, linearise and unionize data
reconr
-21 -22
'reconstructed data for U235 @ 0.0 K'/
9228 /
0.001 /
0 /
-- calculate doppler broadening
broadr
-21 -22 -23
9228 1 /
0.001 /
293.6 /
0 /
-- compute multigroup cross sections
groupr
-21 -23 0 -24
9228 1 0 3 3 1 1 /
'multigroup data for U235'/
293.6 /
10000000000.0 /
4 /
1e-05 0.625 5530.0 821000.0 20000000.0 /
0 /
0 /
stop


In [20]:
# Multiple temperatures and sigma-zero values
deck_multi_temp = (
    InputDeck()
    .moder(nin=20)
    .reconr(isotope="Fe56")
    .broadr(isotope="Fe56", temperatures=[293.6, 600.0, 900.0])
    .groupr(
        isotope="Fe56",
        ign=3,
        iwt=3,
        lord=5,
        temperatures=[293.6, 600.0, 900.0],
        sigz=[1e10, 1e4, 1e3, 1e2, 1e1],
        title="Fe56 multigroup data",
    )
)
print(deck_multi_temp.render())

moder
20 -21
-- reconstruct, linearise and unionize data
reconr
-21 -22
'reconstructed data for Fe56 @ 0.0 K'/
2631 /
0.001 /
0 /
-- calculate doppler broadening
broadr
-21 -22 -23
2631 3 /
0.001 /
293.6 600.0 900.0 /
0 /
-- compute multigroup cross sections
groupr
-21 -23 0 -24
2631 3 0 3 5 3 5 /
'Fe56 multigroup data'/
293.6 600.0 900.0 /
10000000000.0 10000.0 1000.0 100.0 10.0 /
0 /
0 /
stop


In [21]:
# Composite weight function (iwt=4 with eb/tb/ec/tc)
deck_iwt4 = (
    InputDeck()
    .moder(nin=20)
    .reconr(isotope="U235")
    .broadr(isotope="U235", temperatures=[293.6])
    .groupr(
        isotope="U235",
        ign=3,
        iwt=4,
        lord=3,
        temperatures=[293.6],
        sigz=[1e10],
        eb=0.414,
        tb=296.0,
        ec=820.3e3,
        tc=1.4e6,
    )
)
print(deck_iwt4.render())

moder
20 -21
-- reconstruct, linearise and unionize data
reconr
-21 -22
'reconstructed data for U235 @ 0.0 K'/
9228 /
0.001 /
0 /
-- calculate doppler broadening
broadr
-21 -22 -23
9228 1 /
0.001 /
293.6 /
0 /
-- compute multigroup cross sections
groupr
-21 -23 0 -24
9228 3 0 4 3 1 1 /
'multigroup data for U235'/
293.6 /
10000000000.0 /
0.414 296.0 820300.0 1400000.0 /
0 /
0 /
stop


In [22]:
# Error: ign=1 without egn → ValueError
try:
    InputDeck().moder(nin=20).reconr(isotope="U235").broadr(isotope="U235").groupr(
        isotope="U235", ign=1, iwt=3, lord=0,
    )
    print("ERROR: should have raised ValueError")
except ValueError as e:
    print(f"OK: {e}")

# Error: iwt=0 → ValueError
try:
    InputDeck().moder(nin=20).reconr(isotope="U235").broadr(isotope="U235").groupr(
        isotope="U235", ign=3, iwt=0, lord=0,
    )
    print("ERROR: should have raised ValueError")
except ValueError as e:
    print(f"OK: {e}")

# Error: iwt=1 → ValueError
try:
    InputDeck().moder(nin=20).reconr(isotope="U235").broadr(isotope="U235").groupr(
        isotope="U235", ign=3, iwt=1, lord=0,
    )
    print("ERROR: should have raised ValueError")
except ValueError as e:
    print(f"OK: {e}")

OK: ign=1 requires custom neutron group boundaries ('egn').
OK: iwt=0 (read flux from input tape) is not supported.
OK: iwt=1 (tabulated TAB1 weight function) is not supported. Use iwt=2..12 for predefined/analytic weight functions.


In [23]:
# Auto-chaining: verify auto vs manual tape allocation produce identical output
deck_groupr_auto = (
    InputDeck()
    .moder(nin=20)
    .reconr(isotope="U235")
    .broadr(isotope="U235", temperatures=[293.6])
    .groupr(isotope="U235", ign=3, iwt=3, lord=3, temperatures=[293.6], sigz=[1e10])
)

deck_groupr_manual = (
    InputDeck()
    .moder(nin=20, nout=-21)
    .reconr(nendf=-21, npend=-22, isotope="U235")
    .broadr(nendf=-21, nin=-22, nout=-23, isotope="U235", temperatures=[293.6])
    .groupr(nendf=-21, npend=-23, ngout2=-24, isotope="U235", ign=3, iwt=3, lord=3, temperatures=[293.6], sigz=[1e10])
)

assert deck_groupr_auto.render() == deck_groupr_manual.render(), "Auto-chain output differs from manual!"
print("GROUPR auto-chain matches manual deck OK")
print(deck_groupr_auto.render())

GROUPR auto-chain matches manual deck OK
moder
20 -21
-- reconstruct, linearise and unionize data
reconr
-21 -22
'reconstructed data for U235 @ 0.0 K'/
9228 /
0.001 /
0 /
-- calculate doppler broadening
broadr
-21 -22 -23
9228 1 /
0.001 /
293.6 /
0 /
-- compute multigroup cross sections
groupr
-21 -23 0 -24
9228 3 0 3 3 1 1 /
'multigroup data for U235'/
293.6 /
10000000000.0 /
0 /
0 /
stop
